# 13 Directions, places and extremes: circular statistics, Moran's I, variograms, return levels

simplydata's words for directions, places and extreme values (added after 0.2.0), checked on published data:

- `table.find_average_direction_of(full_turn=)` and the Rayleigh test `table.compare_shape_to(shape="even around
  the circle", full_turn=)` on four circular datasets shipped with the R package circular: the homing directions of
  10 green sea turtles (degrees; Luschi et al. 2001), the arrival times of 254 patients at an intensive care unit
  (hours of the day; Cox & Lewis 1966, Fisher 1993 data set B.1), 310 wind directions (radians) and Fisher's (1993)
  data set B.10, three samples of directions (degrees).
- Moran's I, `table.find_autocorrelation_of(x=, y=)`, with 1 / distance weights and with the `nearest=` 4 or 8
  places, on Anselin's (1988) Columbus neighbourhoods (crime, house value, income at the polygon centroids) and the
  Meuse soil samples (log zinc).
- `table.find_variogram_of` on the Meuse samples (log zinc, the gstat vignette's example with its printed fit; log
  copper with the exponential curve; log cadmium).
- `table.fit_distribution_to(distribution="gev" | "generalised pareto")` and `table.find_return_levels_of` on Coles's
  (2001) Port Pirie and Fremantle annual maximum sea levels and his daily rainfall series, with the values he printed.

**Only simplydata runs in this notebook.** The reference values (`references/13_circular_spatial_extremes.csv`, from
`scripts/references/13_circular_spatial_extremes.py`; each check prints its source):
- **published values** (Coles 2001; the gstat vignette), within half a unit of the last printed digit;
- **SciPy** (`circmean`, `circstd`, `directional_stats`; `genextreme` and `genpareto` fitted to a tight stopping
  rule) where SciPy is the parent, and **NumPy written out from the definition** for Moran's I and the empirical
  variogram (an independent calculation);
- **R**: circular (`mean.circular`, `rho.circular`, `sd.circular`, `rayleigh.test`), ape (`Moran.I`), spdep
  (`moran`, `moran.mc`), gstat (`variogram`, `fit.variogram`) and extRemes (`fevd`, `return.level`, `ci.fevd`), run here.

Tolerances (the same for every dataset in this notebook): relative 1e-10 for closed formulas (directions, Moran's I,
semivariances, Rayleigh), relative 1e-6 for fits by iteration (variogram curves, GEV and generalised Pareto, return
levels). The reference fits by iteration are run to a tighter stopping rule than their defaults, which only makes the
comparison stricter. Profile-likelihood intervals are found by extRemes on a grid; they are compared (a) with
extRemes run on the same grid (1e-6) and (b) with the exact ends of the profile found by root finding, within one
step of that grid (stated beside each).

The tables read here are the data files of this repository (the three samples of Fisher B.10 as separate tables,
the Meuse metals on the log scale, the rain excesses above 30 mm and the daily rain with dates), written by the
reference script to `references/data/`.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, versions

NOTEBOOK = "13_circular_spatial_extremes"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def first_row(word, *args, **options):
    return read_row(quiet(word, *args, **options))

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## The average direction: `table.find_average_direction_of`

The average direction (from 0 up to the full turn), the mean resultant length and the circular standard deviation
sqrt(-2 ln R), in the units of the column. The parent: SciPy `circmean`, `directional_stats` and `circstd` with
`low=0, high=full turn`. R circular gives the mean from minus half a turn to half a turn (it is taken modulo the full
turn here) and `sd.circular` in radians (converted here to the column's units).

In [2]:
turtles = load("turtles")
icu = load("icu_arrivals")
wind = load("wind_directions")
fisher_b10 = load("fisher_b10")
turtles.find_average_direction_of("arrival", full_turn=360)

The average direction of "arrival" is 317° (resultant length 0.82: the directions are close together; 1 = all the same direction, 0 = no average direction). Circular standard deviation 36° (10 values).
Method: SciPy stats.circmean, circstd and circvar with low=0, high=360, and directional_stats for the resultant length; the same numbers as R circular's mean.circular, rho.circular, sd.circular and var.circular with the units given.
simplydata chooses: the average from 0 up to 360 (R circular gives the same direction from -180 to 180, such as -43° for 317°), and the circular standard deviation sqrt(-2 ln(resultant length)), as R sd.circular, in the units of the column.


,rows,average direction,resultant length,circular standard deviation,circular variance
1,10,317.23532,0.81865,36.246008,0.18135


In [3]:
DIRECTION_SETS = [("turtles (circular)", turtles, "arrival", 360),
                  ("ICU arrivals (circular fisherB1)", icu, "hour", 24),
                  ("wind (circular)", wind, "direction_radians", "radians")]
DIRECTION_SETS += [(f"Fisher B.10 {s} (circular fisherB10)", load(f"fisher_b10_{s}"), "direction", 360)
                   for s in ("set1", "set2", "set3")]
QUANTITIES = ["average direction", "resultant length", "circular standard deviation"]

for dataset, data, column, full_turn in DIRECTION_SETS:
    row = first_row(data.find_average_direction_of, column, full_turn=full_turn)
    checks.check_values(dataset, "find_average_direction_of", {quantity: row[quantity] for quantity in QUANTITIES})
# one call with groups gives the same as each group on its own
for row in read_rows(quiet(fisher_b10.find_average_direction_of, "direction", full_turn=360, for_each_group_in="set")):
    checks.check_values(f"Fisher B.10 {row['set']} (circular fisherB10)", "find_average_direction_of (for_each_group_in=)",
                        {quantity: row[quantity] for quantity in QUANTITIES})

agrees  turtles (circular) | find_average_direction_of | average direction: simplydata 317.235319509, SciPy circmean, directional_stats, circstd 317.235319509; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy circmean, directional_stats, circstd (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.circmean.html
agrees  turtles (circular) | find_average_direction_of | average direction: simplydata 317.235319509, R circular 317.235319509; relative difference 1.79e-16 (allowed 1e-10)
      Source: R circular (run reference) - https://cran.r-project.org/package=circular (version 0.5.2)
agrees  turtles (circular) | find_average_direction_of | resultant length: simplydata 0.818649629674, SciPy circmean, directional_stats, circstd 0.818649629674; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy circmean, directional_stats, circstd (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.circm

## The Rayleigh test: `table.compare_shape_to(shape="even around the circle", full_turn=)`

The statistic is the mean resultant length, checked against SciPy's; the statistic and the p-value are checked
against R circular `rayleigh.test` with the units given (without them R reads degrees as radians).

In [4]:
icu.compare_shape_to("hour", shape="even around the circle", full_turn=24)

The directions of "hour" have a preferred direction (Rayleigh test: resultant length 0.32, 254 values).
If the directions were spread evenly around the circle, directions this close together or closer would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001).
Test: the Rayleigh test, R circular::rayleigh.test(circular::circular(x, units = "hours", template = "none")) (the units are always given: R circular reads plain numbers as radians).


,test,what it compares,statistic name,statistic,p-value,rows
1,Rayleigh test,"the directions of ""hour"" with directions sprea...",resultant length,0.316352,9.125844e-12,254


In [5]:
for dataset, data, column, full_turn in DIRECTION_SETS:
    row = first_row(data.compare_shape_to, column, shape="even around the circle", full_turn=full_turn)
    checks.check_values(dataset, "compare_shape_to (Rayleigh)", {"resultant length": row["statistic"],
                                                                 "p-value": row["p-value"]})

agrees  turtles (circular) | compare_shape_to (Rayleigh) | resultant length: simplydata 0.818649629674, SciPy directional_stats 0.818649629674; relative difference 1.36e-16 (allowed 1e-10)
      Source: SciPy directional_stats (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.directional_stats.html
agrees  turtles (circular) | compare_shape_to (Rayleigh) | resultant length: simplydata 0.818649629674, R circular rayleigh.test 0.818649629674; relative difference 0.00e+00 (allowed 1e-10)
      Source: R circular rayleigh.test (run reference) - https://search.r-project.org/CRAN/refmans/circular/html/rayleigh.test.html (version 0.5.2)
agrees  turtles (circular) | compare_shape_to (Rayleigh) | p-value: simplydata 0.000305399546463, R circular rayleigh.test 0.000305399546463; relative difference 6.18e-14 (allowed 1e-10)
      Source: R circular rayleigh.test (run reference) - https://search.r-project.org/CRAN/refmans/circular/html/rayleigh.test.html (ve

## Moran's I: `table.find_autocorrelation_of(x=, y=)`

Moran's I = (n / S0) z'Wz / z'z, with z the values minus their mean and W the weights with each row scaled to sum
to 1 (so S0 = n). simplydata's default weights are 1 / distance between all pairs; `nearest=k` uses the k nearest
places. The independent calculation is this formula in NumPy; R references are `ape::Moran.I` (1 / distance weights,
which it scales by row itself) and `spdep::moran` on `knn2nb(knearneigh())` with `style = "W"`. The p-value from 999
random shufflings is compared with `spdep::moran.mc(nsim = 999)` after `set.seed(1)` (simplydata's `with_seed=1`);
the value Moran's I takes with no pattern is -1 / (n - 1).

Columbus (Anselin 1988) is analysed in the spdep vignette with contiguity (shared border) weights, which
simplydata does not offer; it is used here with distances between the polygon centroids.

In [6]:
columbus = load("columbus")
meuse = load("meuse")
columbus.find_autocorrelation_of("crime", x="x", y="y")

Nearby places have more alike values of "crime" than distant ones: Moran's I = 0.165 (no pattern gives about 0, here -0.021), where every pair counts, nearer pairs more (weights 1 / distance, each row's weights summing to 1).
If the values were scattered over the places at random, nearby values at least this alike would turn up in at most about 1 of 1000 studies like this one (p = 0.001, the smallest p that 999 shuffles can give).
Method: R pts <- sf::st_as_sf(data.frame(x = x, y = y), coords = c("x", "y")); nb <- spdep::dnearneigh(pts, 0, Inf); lw <- spdep::nb2listwdist(nb, pts, type = "idw", alpha = 1, style = "W"); set.seed(1); m <- spdep::moran.mc(z, lw, nsim = 999); (49 places; spdep::moran.test for the value with no pattern, and with alternative = "less" before a sentence about nearby places being less alike).
simplydata chooses: the 1 / distance weights between all pairs, row-standardised, 999 permutations after set.seed(1) and a one-sided test (nearby places more alike), spdep'

,Moran's I,value with no pattern,p-value,permutations,places,weights
1,0.16528,-0.020833,0.001,999,49,"1 / distance, all pairs"


In [7]:
MORAN_SETS = [("Columbus (spData)", columbus, c) for c in ("crime", "house_value", "income")]
MORAN_SETS += [("Meuse (sp)", meuse, "log_zinc")]
for dataset, data, column in MORAN_SETS:
    for nearest in (None, 4, 8):
        weights = "1 / distance" if nearest is None else f"nearest={nearest}"
        row = first_row(data.find_autocorrelation_of, column, x="x", y="y", nearest=nearest)
        checks.check_values(dataset, "find_autocorrelation_of (x=, y=)",
                            {f"{column}: Moran's I ({weights})": row["Moran's I"],
                             f"{column}: value with no pattern ({weights})": row["value with no pattern"],
                             f"{column}: p-value, 999 shuffles ({weights})": row["p-value"]})

agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | crime: Moran's I (1 / distance): simplydata 0.165279918722, NumPy, the formula 0.165279918722; relative difference 3.36e-16 (allowed 1e-10)
      Source: NumPy, the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | crime: Moran's I (1 / distance): simplydata 0.165279918722, R ape Moran.I 0.165279918722; relative difference 3.36e-16 (allowed 1e-10)
      Source: R ape Moran.I (run reference) - https://search.r-project.org/CRAN/refmans/ape/html/Moran.I.html (version 5.8.1)
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | crime: value with no pattern (1 / distance): simplydata -0.0208333333333, -1 / (n - 1) -0.0208333333333; relative difference 1.50e-15 (allowed 1e-10)
      Source: -1 / (n - 1) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blo

agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | crime: Moran's I (nearest=4): simplydata 0.624933667352, NumPy, the formula 0.624933667352; relative difference 0.00e+00 (allowed 1e-10)
      Source: NumPy, the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | crime: Moran's I (nearest=4): simplydata 0.624933667352, R spdep moran (knn2nb) 0.624933667352; relative difference 0.00e+00 (allowed 1e-10)
      Source: R spdep moran (knn2nb) (run reference) - https://search.r-project.org/CRAN/refmans/spdep/html/moran.html (version 1.4.3)
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | crime: value with no pattern (nearest=4): simplydata -0.0208333333333, -1 / (n - 1) -0.0208333333333; relative difference 1.50e-15 (allowed 1e-10)
      Source: -1 / (n - 1) (run reference) - https://github.com/bioforkgithub/simplydata_

agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | house_value: Moran's I (1 / distance): simplydata 0.029140762388, NumPy, the formula 0.029140762388; relative difference 2.74e-15 (allowed 1e-10)
      Source: NumPy, the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | house_value: Moran's I (1 / distance): simplydata 0.029140762388, R ape Moran.I 0.029140762388; relative difference 2.74e-15 (allowed 1e-10)
      Source: R ape Moran.I (run reference) - https://search.r-project.org/CRAN/refmans/ape/html/Moran.I.html (version 5.8.1)
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | house_value: value with no pattern (1 / distance): simplydata -0.0208333333333, -1 / (n - 1) -0.0208333333333; relative difference 1.50e-15 (allowed 1e-10)
      Source: -1 / (n - 1) (run reference) - https://github.com/bioforkgithub/si

agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | income: Moran's I (1 / distance): simplydata 0.110987430036, NumPy, the formula 0.110987430036; relative difference 7.50e-16 (allowed 1e-10)
      Source: NumPy, the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | income: Moran's I (1 / distance): simplydata 0.110987430036, R ape Moran.I 0.110987430036; relative difference 1.25e-16 (allowed 1e-10)
      Source: R ape Moran.I (run reference) - https://search.r-project.org/CRAN/refmans/ape/html/Moran.I.html (version 5.8.1)
agrees  Columbus (spData) | find_autocorrelation_of (x=, y=) | income: value with no pattern (1 / distance): simplydata -0.0208333333333, -1 / (n - 1) -0.0208333333333; relative difference 1.50e-15 (allowed 1e-10)
      Source: -1 / (n - 1) (run reference) - https://github.com/bioforkgithub/simplydata_tests/

agrees  Meuse (sp) | find_autocorrelation_of (x=, y=) | log_zinc: Moran's I (1 / distance): simplydata 0.119126247659, NumPy, the formula 0.119126247659; relative difference 5.82e-16 (allowed 1e-10)
      Source: NumPy, the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Meuse (sp) | find_autocorrelation_of (x=, y=) | log_zinc: Moran's I (1 / distance): simplydata 0.119126247659, R ape Moran.I 0.119126247659; relative difference 5.82e-16 (allowed 1e-10)
      Source: R ape Moran.I (run reference) - https://search.r-project.org/CRAN/refmans/ape/html/Moran.I.html (version 5.8.1)
agrees  Meuse (sp) | find_autocorrelation_of (x=, y=) | log_zinc: value with no pattern (1 / distance): simplydata -0.00649350649351, -1 / (n - 1) -0.00649350649351; relative difference 1.44e-14 (allowed 1e-10)
      Source: -1 / (n - 1) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scr

agrees  Meuse (sp) | find_autocorrelation_of (x=, y=) | log_zinc: Moran's I (nearest=8): simplydata 0.489606337219, NumPy, the formula 0.489606337219; relative difference 3.40e-16 (allowed 1e-10)
      Source: NumPy, the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Meuse (sp) | find_autocorrelation_of (x=, y=) | log_zinc: Moran's I (nearest=8): simplydata 0.489606337219, R spdep moran (knn2nb) 0.489606337219; relative difference 1.13e-16 (allowed 1e-10)
      Source: R spdep moran (knn2nb) (run reference) - https://search.r-project.org/CRAN/refmans/spdep/html/moran.html (version 1.4.3)
agrees  Meuse (sp) | find_autocorrelation_of (x=, y=) | log_zinc: value with no pattern (nearest=8): simplydata -0.00649350649351, -1 / (n - 1) -0.00649350649351; relative difference 1.44e-14 (allowed 1e-10)
      Source: -1 / (n - 1) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob

## The variogram: `table.find_variogram_of`

The empirical variogram: all pairs of places in 15 distance bins up to a third of the diagonal of the bounding box
(gstat's defaults); for each bin the number of pairs, their average distance and the semivariance (half the average
squared difference). It is written out in NumPy here as an independent calculation, and compared with R
`gstat::variogram`. The fitted curve (nugget, partial sill, range) is compared with R `gstat::fit.variogram(v,
vgm(model))` from gstat's own start values (as simplydata documents) and, for the spherical curve of log zinc, with
the fit printed in the gstat vignette (Pebesma, "gstat: spatial and spatio-temporal geostatistical modelling":
nugget 0.0507, partial sill 0.5906, range 897; there fitted from `vgm(1, "Sph", 900, 1)`).

In [8]:
meuse.find_variogram_of("log_zinc", x="x", y="y")

Values of "log_zinc" become unrelated beyond about 897 (the range, in the units of "x" and "y").
Nugget 0.0507 (differences even between very close places: measurement error or variation at shorter distances than were sampled), partial sill 0.591 (the further rise from the nugget up to the plateau).
Method: R gstat::variogram(z ~ 1, locations = ~ x + y, data) (15 distance bins, up to a third of the diagonal), then gstat::fit.variogram(v, gstat::vgm("Sph")) with gstat's own start values and fit.method = 7 (weighted least squares, the weights the number of pairs / distance^2).
simplydata chooses: the spherical curve (the default), gstat's own start values, and the range said in words, because for the exponential, Gaussian and Matern curves gstat's range parameter is not the practical range.


,distance,semivariance,pairs,fitted semivariance,nugget,partial sill,range
1,79.292437,0.123448,57,0.128768,0.05066,0.590605,897.001083
2,163.973666,0.216218,299,0.210802,0.05066,0.590605,897.001083
3,267.364828,0.302786,419,0.306898,0.05066,0.590605,897.001083
4,372.735422,0.412145,457,0.397597,0.05066,0.590605,897.001083
5,478.476695,0.463413,547,0.478399,0.05066,0.590605,897.001083
6,585.340581,0.564693,533,0.546705,0.05066,0.590605,897.001083
7,693.145256,0.568968,574,0.598975,0.05066,0.590605,897.001083
8,796.183649,0.618677,564,0.630493,0.05066,0.590605,897.001083
9,903.146498,0.647148,589,0.641265,0.05066,0.590605,897.001083
10,1011.291773,0.691570,543,0.641265,0.05066,0.590605,897.001083


In [9]:
for column, kind in [("log_zinc", "spherical"), ("log_copper", "exponential"), ("log_cadmium", "spherical")]:
    found = read_rows(quiet(meuse.find_variogram_of, column, x="x", y="y", of_kind=kind))
    values = {}
    for k, row in enumerate(found):
        for quantity in ("pairs", "distance", "semivariance"):
            values[f"{column}: bin {k + 1} {quantity}"] = row[quantity]
    for quantity in ("nugget", "partial sill", "range"):
        values[f"{column}: {quantity}"] = found[0][quantity]
    checks.check_values("Meuse (sp)", f"find_variogram_of ({kind})", values)

agrees  Meuse (sp) | find_variogram_of (spherical) | log_zinc: bin 1 pairs: simplydata 57, NumPy, the definition 57; relative difference 0.00e+00 (allowed 1e-10)
      Source: NumPy, the definition (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Meuse (sp) | find_variogram_of (spherical) | log_zinc: bin 1 pairs: simplydata 57, R gstat variogram 57; relative difference 0.00e+00 (allowed 1e-10)
      Source: R gstat variogram (run reference) - https://search.r-project.org/CRAN/refmans/gstat/html/variogram.html (version 2.1.6)
agrees  Meuse (sp) | find_variogram_of (spherical) | log_zinc: bin 1 distance: simplydata 79.2924374558, NumPy, the definition 79.2924374558; relative difference 0.00e+00 (allowed 1e-10)
      Source: NumPy, the definition (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Meuse (sp) | find_var

## Extremes: the GEV and generalised Pareto fits, `table.fit_distribution_to`

Coles (2001) printed, for the annual maximum sea levels at **Port Pirie** (section 3.4.1) mu = 3.87, sigma = 0.198,
xi = -0.050 with standard errors 0.028, 0.020 and 0.098, and for **Fremantle** (stationary model, section 6.3.1)
mu = 1.482, sigma = 0.141, xi = -0.217. For his **daily rainfall** above 30 mm (section 4.4.1) the generalised
Pareto fit of the excesses is sigma = 7.44, xi = 0.184.

The parent: SciPy `genextreme.fit` (its shape c is -xi) and `genpareto.fit(floc=0)`, here with a Nelder-Mead search
run to xtol = ftol = 1e-12 from SciPy's own start. R: `extRemes::fevd(type = "GEV" | "GP", method = "MLE")` with
`optim`'s `reltol = 1e-14`; the standard errors from the inverse of its Hessian (optim's finite differences). simplydata's standard errors also come from a Hessian found by finite
differences, so these two agree only as far as two such approximations do (an attempt to get an exact reference with
`numDeriv::hessian` gave standard errors that moved in the fourth digit with its step size on Fremantle, so none is
used).

In [10]:
portpirie = load("portpirie")
fremantle = load("fremantle")
excess = load("rain_excess")
daily = load("rain_daily")
portpirie.fit_distribution_to("sea_level", distribution="gev").turn_into_table()

Fitted a GEV distribution to "sea_level" (65 values): location = 3.87 (standard error 0.0279), scale = 0.198 (0.0202), shape = -0.0501 (0.0983). Log-likelihood 4.3, AIC -2.7.
Method: maximum likelihood, as extRemes::fevd(x, type = "GEV", method = "MLE"). The values were found by a numerical search (Nelder-Mead, then BFGS). To check whether the shape fits, use table.compare_shape_to("sea_level", "gev").


,distribution,column,location,scale,shape,standard error of location,standard error of scale,standard error of shape,log-likelihood,AIC,BIC,rows,held fixed,method,settled
1,gev,sea_level,3.87475,0.198044,-0.05011,0.027932,0.020249,0.098256,4.339058,-2.678117,3.845045,65,,maximum likelihood (numerical search (Nelder-M...,yes


In [11]:
GEV = ["location", "scale", "shape", "standard error of location", "standard error of scale", "standard error of shape"]
for dataset, data in (("Port Pirie (ismev)", portpirie), ("Fremantle (ismev)", fremantle)):
    row = read_row(quiet(data.fit_distribution_to, "sea_level", distribution="gev").turn_into_table())
    checks.check_values(dataset, 'fit_distribution_to (distribution="gev")', {quantity: row[quantity] for quantity in GEV})
row = read_row(quiet(excess.fit_distribution_to, "excess", distribution="generalised pareto").turn_into_table())
checks.check_values("Coles rain (ismev)", 'fit_distribution_to (distribution="generalised pareto")',
                    {"scale": row["scale"], "shape": row["shape"]})

agrees  Port Pirie (ismev) | fit_distribution_to (distribution="gev") | location: simplydata 3.87474985466, Coles (2001) 3.4.1, printed 3.87; difference 4.75e-03 (allowed 5e-03)
      Source: Coles (2001) 3.4.1, printed (printed) - https://doi.org/10.1007/978-1-4471-3675-0
agrees  Port Pirie (ismev) | fit_distribution_to (distribution="gev") | location: simplydata 3.87474985466, SciPy genextreme.fit (tight) 3.87474985805; relative difference 8.74e-10 (allowed 1e-06)
      Source: SciPy genextreme.fit (tight) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.genextreme.html
agrees  Port Pirie (ismev) | fit_distribution_to (distribution="gev") | location: simplydata 3.87474985466, R extRemes fevd 3.87474985509; relative difference 1.11e-10 (allowed 1e-06)
      Source: R extRemes fevd (run reference) - https://search.r-project.org/CRAN/refmans/extRemes/html/fevd.html (version 2.2.1)
agrees  Port Pirie (ismev) | fit_distribution_to (distribution="ge

agrees  Fremantle (ismev) | fit_distribution_to (distribution="gev") | location: simplydata 1.48234176831, Coles (2001) 6.3.1, printed 1.482; difference 3.42e-04 (allowed 5e-04)
      Source: Coles (2001) 6.3.1, printed (printed) - https://doi.org/10.1007/978-1-4471-3675-0
agrees  Fremantle (ismev) | fit_distribution_to (distribution="gev") | location: simplydata 1.48234176831, SciPy genextreme.fit (tight) 1.48234177029; relative difference 1.33e-09 (allowed 1e-06)
      Source: SciPy genextreme.fit (tight) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.genextreme.html
agrees  Fremantle (ismev) | fit_distribution_to (distribution="gev") | location: simplydata 1.48234176831, R extRemes fevd 1.48234176866; relative difference 2.30e-10 (allowed 1e-06)
      Source: R extRemes fevd (run reference) - https://search.r-project.org/CRAN/refmans/extRemes/html/fevd.html (version 2.2.1)
agrees  Fremantle (ismev) | fit_distribution_to (distribution="gev")

6

## Return levels: `table.find_return_levels_of`

The level reached on average once in T years: for the GEV, z = mu - sigma / xi (1 - (-ln(1 - 1/T))^(-xi)); above a
threshold u (generalised Pareto, 365.25 days a year, zeta the share of days above u),
z = u + sigma / xi ((T 365.25 zeta)^xi - 1). Coles printed for Port Pirie the 10- and 100-year levels 4.30 and 4.69,
with the normal-approximation interval 4.38 to 5.00 for the 100-year level, and for the daily rain the 100-year level
106.3. The levels are compared with these values, with the formula applied to the R fit above, and with R
`extRemes::return.level` (on the tight fit, and on extRemes's default fit, as simplydata calls it).

The profile-likelihood interval (simplydata's main interval) is found by `extRemes::ci.fevd(method = "proflik")` on
a grid of 200 levels from the low end of the normal interval to two of its widths above its high end (simplydata's
documented search). It is compared with extRemes on that grid (1e-6), and with the exact ends of the profile found
here by root finding (the model refitted with the return level as one of its parameters; the ends are where twice
the drop in log-likelihood equals the chi-squared 95 % point, 3.841), allowing one step of that grid.

In [12]:
portpirie.find_return_levels_of("sea_level", once_in=[10, 100], sorted_by="year")

GEV (generalised extreme value) curve fitted to 65 block maxima of "sea_level" (each row the largest value of one year).
A level of 4.30 is reached on average once in 10 years (a 1-in-10 chance in any one year); 95 % CI 4.21 to 4.44 (profile likelihood).
A level of 4.69 is reached on average once in 100 years (a 1-in-100 chance in any one year); 95 % CI 4.49 to 5.26 (profile likelihood).
Shape xi = -0.050 (95 % CI -0.243 to 0.142, normal approximation): its interval includes 0, so the upper tail is close to a Gumbel curve. Location 3.87, scale 0.198. (xi in the sign of extRemes and Coles 2001; SciPy's genextreme uses c = -xi.)
Method: R extRemes::fevd(x, type = "GEV", method = "MLE"); extRemes::return.level(); extRemes::ci.fevd(type = "return.level", method = "proflik") for each interval.
simplydata chooses: maximum likelihood; profile-likelihood intervals (Coles 2001, section 3.3.3: the normal approximation is too narrow and too symmetric for long return periods); no declustering. The

,once in,time unit,level,95 % CI low,95 % CI high,interval,normal 95 % CI low,normal 95 % CI high
1,10.0,years,4.296212,4.205996,4.444549,profile likelihood,4.188385,4.404039
2,100.0,years,4.688404,4.494984,5.257408,profile likelihood,4.377125,4.999682


In [13]:
def check_return_levels(dataset, method, found):
    for T, row in found.items():
        values = {f"{T:g}-year {quantity}": row[quantity] for quantity in ("level", "normal 95 % CI low", "normal 95 % CI high")}
        values.update({f"{T:g}-year profile {quantity}": row[quantity] for quantity in ("95 % CI low", "95 % CI high")})
        checks.check_values(dataset, method, values)


found = {row["once in"]: row for row in read_rows(quiet(portpirie.find_return_levels_of, "sea_level", once_in=[10, 100],
                                                        sorted_by="year"))}
check_return_levels("Port Pirie (ismev)", "find_return_levels_of", found)

quiet(daily.turn_into_dates, "date")
found = {row["once in"]: row for row in read_rows(quiet(daily.find_return_levels_of, "rain", once_in=[10, 100],
                                                        values_above=30, sorted_by="date"))}
check_return_levels("Coles rain (ismev)", "find_return_levels_of (values_above=30)", found)

agrees  Port Pirie (ismev) | find_return_levels_of | 10-year level: simplydata 4.29621194427, Coles (2001) 3.4.1, printed 4.3; difference 3.79e-03 (allowed 5e-03)
      Source: Coles (2001) 3.4.1, printed (printed) - https://doi.org/10.1007/978-1-4471-3675-0
agrees  Port Pirie (ismev) | find_return_levels_of | 10-year level: simplydata 4.29621194427, R extRemes return.level (default stopping rule) 4.29621194427; relative difference 0.00e+00 (allowed 1e-06)
      Source: R extRemes return.level (default stopping rule) (run reference) - https://search.r-project.org/CRAN/refmans/extRemes/html/return.level.html (version 2.2.1)
agrees  Port Pirie (ismev) | find_return_levels_of | 10-year level: simplydata 4.29621194427, the formula on the R fit 4.29621193824; relative difference 1.40e-09 (allowed 1e-06)
      Source: the formula on the R fit (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/13_circular_spatial_extremes.py
agrees  Port Pirie (is

explained difference – precision limit (C29)  Coles rain (ismev) | find_return_levels_of (values_above=30) | 10-year level: simplydata 65.9615792999, R extRemes return.level (default stopping rule) 65.9614236348; relative difference 2.36e-06 (allowed 1e-06)
      why: References run with R's default stopping rule; simplydata now converges tighter (its A7 fix) - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c29-references-run-with-rs-default-stopping-rule-simplydata-now-converges-tighter-its-a7-fix
      Source: R extRemes return.level (default stopping rule) (run reference) - https://search.r-project.org/CRAN/refmans/extRemes/html/return.level.html (version 2.2.1)
agrees  Coles rain (ismev) | find_return_levels_of (values_above=30) | 10-year level: simplydata 65.9615792999, the formula on the R fit 65.9615792999; relative difference 0.00e+00 (allowed 1e-06)
      Source: the formula on the R fit (run reference) - https://github.com/bioforkgithub/simplydata_tes

## Result

In [14]:
results = checks.finish()

13_circular_spatial_extremes: 467 comparisons - 460 agrees, 7 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
